# Building a Semantic Search Engine with Vector Databases

Ever wished you could search through documents by *meaning* rather than just keywords? That's what vector databases enable!

In this tutorial, we'll build a **semantic search engine for research papers** using:
- 🔤 **Sentence Transformers** for converting text to embeddings
- 🚀 **FAISS** (Facebook AI Similarity Search) for fast vector search
- 📊 **Real arXiv papers** as our dataset

## What You'll Learn

1. How text embeddings capture semantic meaning
2. Building and querying a vector database
3. Creating a realistic paper search application
4. Understanding similarity metrics
5. Scaling to thousands of documents

Let's dive in!

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from typing import List, Tuple
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All imports successful!")

## Step 1: Create a Realistic Dataset

Let's create a dataset of research paper abstracts from different fields:
- 🧬 **Biology/Medicine**
- 🤖 **AI/Machine Learning**
- 🌍 **Climate Science**
- 📈 **Economics**

In [ ]:
# Realistic research paper abstracts
papers = [
    {
        "title": "CRISPR-Cas9 Gene Editing in Cancer Therapy",
        "abstract": "We present a novel approach using CRISPR-Cas9 technology to target oncogenes in lung cancer cells. Our results demonstrate a 60% reduction in tumor growth in mouse models through precise genome editing.",
        "field": "Biology"
    },
    {
        "title": "Transformer Models for Protein Structure Prediction",
        "abstract": "Deep learning transformers can predict protein 3D structures from amino acid sequences with unprecedented accuracy. We achieve RMSD scores comparable to AlphaFold2 using a lighter architecture.",
        "field": "AI-Biology"
    },
    {
        "title": "Climate Tipping Points in the Arctic",
        "abstract": "Analysis of ice core data reveals accelerating permafrost thaw in the Arctic region. We identify critical temperature thresholds that could trigger irreversible methane release and feedback loops.",
        "field": "Climate"
    },
    {
        "title": "Deep Reinforcement Learning for Robotics",
        "abstract": "We introduce a sample-efficient RL algorithm for robotic manipulation tasks. Our approach combines model-based planning with learned world models to achieve human-level performance on complex assembly tasks.",
        "field": "AI-Robotics"
    },
    {
        "title": "Neural Architecture Search with Evolutionary Algorithms",
        "abstract": "Automated neural architecture search using genetic algorithms discovers novel CNN architectures that outperform hand-designed networks on ImageNet classification while requiring 40% fewer parameters.",
        "field": "AI"
    },
    {
        "title": "Microbiome Diversity and Immune Response",
        "abstract": "Gut microbiome composition strongly correlates with immune system function. We identify 15 bacterial species associated with enhanced T-cell response and reduced inflammation in clinical trials.",
        "field": "Biology"
    },
    {
        "title": "Economic Impact of Universal Basic Income",
        "abstract": "Large-scale field experiments in Kenya demonstrate that universal basic income programs increase entrepreneurship by 23% and improve mental health outcomes without reducing labor force participation.",
        "field": "Economics"
    },
    {
        "title": "Machine Learning for Climate Model Emulation",
        "abstract": "Neural networks can emulate expensive climate models 10000x faster while maintaining accuracy. This enables rapid exploration of emission scenarios and uncertainty quantification in climate projections.",
        "field": "AI-Climate"
    },
    {
        "title": "Graph Neural Networks for Drug Discovery",
        "abstract": "We apply graph neural networks to predict molecular properties and drug-target binding affinity. Our model identifies 12 promising candidates for Alzheimer's treatment from a library of 1 million compounds.",
        "field": "AI-Biology"
    },
    {
        "title": "Carbon Capture Technologies and Scalability",
        "abstract": "Direct air capture using novel metal-organic frameworks achieves 95% CO2 removal efficiency. Economic analysis suggests scalability to gigatonne levels by 2035 with current cost trajectories.",
        "field": "Climate"
    },
    {
        "title": "Behavioral Economics of Climate Action",
        "abstract": "Nudge interventions combining social norms and financial incentives increase household solar adoption by 40%. We analyze psychological barriers to climate action across 50,000 participants.",
        "field": "Economics"
    },
    {
        "title": "Immunotherapy Checkpoint Inhibitors for Melanoma",
        "abstract": "Combined PD-1 and CTLA-4 blockade shows 70% response rate in advanced melanoma patients. Biomarker analysis reveals tumor mutation burden as the strongest predictor of treatment efficacy.",
        "field": "Biology"
    },
    {
        "title": "Large Language Models for Code Generation",
        "abstract": "We train a 20B parameter transformer on code repositories to generate functionally correct programs from natural language descriptions. The model passes 85% of LeetCode medium problems.",
        "field": "AI"
    },
    {
        "title": "Ocean Acidification and Coral Reef Resilience",
        "abstract": "Rising ocean acidity reduces coral calcification rates by 30%. However, certain coral species show adaptive capacity through symbiont shuffling and heat shock protein expression.",
        "field": "Climate"
    },
    {
        "title": "Inequality and Economic Growth Dynamics",
        "abstract": "Panel data from 150 countries shows an inverted-U relationship between inequality and growth. Moderate inequality stimulates innovation while extreme inequality hinders human capital accumulation.",
        "field": "Economics"
    },
    {
        "title": "Self-Supervised Learning for Medical Imaging",
        "abstract": "Contrastive learning on unlabeled X-ray images creates representations that transfer to downstream tasks. Our approach matches supervised baselines using only 10% labeled data for pneumonia detection.",
        "field": "AI-Biology"
    },
    {
        "title": "mRNA Vaccine Technology and Rapid Development",
        "abstract": "Lipid nanoparticle delivery of mRNA enables rapid vaccine development for emerging pathogens. We demonstrate protective immunity against influenza variants within 6 weeks from sequence to first dose.",
        "field": "Biology"
    },
    {
        "title": "Federated Learning for Privacy-Preserving AI",
        "abstract": "Decentralized model training across hospital networks enables collaborative learning without sharing patient data. Our federated approach achieves centralized performance while maintaining differential privacy.",
        "field": "AI"
    },
]

# Extract into lists
titles = [p['title'] for p in papers]
abstracts = [p['abstract'] for p in papers]
fields = [p['field'] for p in papers]

print(f\"📚 Loaded {len(papers)} research papers across {len(set(fields))} fields\")\nprint(f\"\\nFields: {', '.join(sorted(set(fields)))}\")

## Step 2: Generate Embeddings

We'll use a **sentence transformer** to convert each abstract into a dense vector that captures its semantic meaning.

In [ ]:
# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded: {model.get_sentence_embedding_dimension()} dimensions\n")

# Generate embeddings for all abstracts
embeddings = model.encode(abstracts, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

print(f"\n✅ Generated embeddings: {embeddings.shape}")
print(f"   Each abstract is now a {embeddings.shape[1]}-dimensional vector!")

## Key Takeaways

### What We Built

✅ A **semantic search engine** that understands meaning, not just keywords  
✅ Explored **3 different vector database implementations**  
✅ Learned about **embeddings** and **similarity metrics**  
✅ Compared **performance** across different approaches  

### When to Use Each Database

| Database | Best For | Pros | Cons |
|----------|----------|------|------|
| **FAISS** | Production, large-scale | Blazing fast, GPU support, approximate search | More complex API |
| **ChromaDB** | Prototyping, small-medium scale | Easy API, built-in metadata | Slower than FAISS |
| **NumPy** | Learning, tiny datasets | Simple, educational | Not scalable |

### Real-World Applications

- 🔍 **Semantic search** (like we built!)
- 🤖 **RAG systems** (Retrieval-Augmented Generation for LLMs)
- 🎯 **Recommendation engines**
- 🖼️ **Image/video similarity search**
- 📊 **Anomaly detection**
- 🧬 **Drug discovery** (molecular similarity)

### Next Steps

1. **Scale up**: Try with 10,000+ documents
2. **Add metadata filtering**: Filter by date, author, journal
3. **Implement approximate search**: Use FAISS IVF indices for massive datasets
4. **Build a web API**: Wrap this in FastAPI/Flask
5. **Add hybrid search**: Combine semantic + keyword search

### Resources

- [FAISS Documentation](https://github.com/facebookresearch/faiss)
- [ChromaDB Docs](https://docs.trychroma.com/)
- [Sentence Transformers](https://www.sbert.net/)
- [Pinecone Tutorial](https://www.pinecone.io/learn/vector-database/)

---

**🎉 Congratulations!** You now understand how vector databases power modern AI applications!

In [ ]:
import time

def benchmark_search(n_queries=100):
    """Benchmark search performance across different implementations."""
    
    # Generate random queries
    query_texts = [
        "machine learning",
        "climate change", 
        "gene therapy",
        "economic growth",
        "neural networks"
    ]
    
    timings = {'FAISS': [], 'NumPy': []}
    
    for _ in range(n_queries):
        query = np.random.choice(query_texts)
        query_embedding = model.encode([query])[0].astype('float32')
        
        # FAISS timing
        start = time.time()
        faiss_index.search(query_embedding.reshape(1, -1), 5)
        timings['FAISS'].append(time.time() - start)
        
        # NumPy timing
        start = time.time()
        simple_db.search(query_embedding, k=5)
        timings['NumPy'].append(time.time() - start)
    
    # Results
    results = pd.DataFrame({
        'Database': ['FAISS', 'NumPy'],
        'Avg Time (ms)': [
            np.mean(timings['FAISS']) * 1000,
            np.mean(timings['NumPy']) * 1000
        ],
        'Std Dev (ms)': [
            np.std(timings['FAISS']) * 1000,
            np.std(timings['NumPy']) * 1000
        ]
    })
    
    print(f"\n⚡ Performance Benchmark ({n_queries} queries)\n")
    print(results.to_string(index=False))
    print(f"\n💡 FAISS is ~{np.mean(timings['NumPy']) / np.mean(timings['FAISS']):.1f}x faster!")
    
    # Visualization
    plt.figure(figsize=(10, 5))
    plt.bar(['FAISS', 'NumPy'], 
            [np.mean(timings['FAISS']) * 1000, np.mean(timings['NumPy']) * 1000],
            color=['#4ECDC4', '#FF6B6B'],
            alpha=0.8,
            edgecolor='black',
            linewidth=2)
    plt.ylabel('Average Query Time (ms)', fontsize=12)
    plt.title('Vector Database Performance Comparison', fontsize=14, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

benchmark_search(n_queries=100)

## Performance Comparison

Let's compare the performance of different vector databases as we scale up!

In [ ]:
# Query 3: Economic impacts
search_engine.display_results("economic policy and social welfare", k=3)

In [ ]:
# Query 2: Filtered search
search_engine.display_results("improving human health", k=3, field_filter="Biology")

In [ ]:
# Query 1: General search
search_engine.display_results("how can AI help solve climate change?", k=3)

### Try Some Interesting Queries!

In [ ]:
class PaperSearchEngine:
    """A complete semantic search engine for research papers."""
    
    def __init__(self, papers, embeddings, model):
        self.papers = papers
        self.embeddings = embeddings
        self.model = model
        
        # Create FAISS index
        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        # Normalize for cosine similarity
        self.embeddings_norm = embeddings / np.linalg.norm(
            embeddings, axis=1, keepdims=True
        )
        self.index.add(self.embeddings_norm)
    
    def search(self, query: str, k: int = 5, field_filter: str = None):
        """
        Search papers with optional field filtering.
        
        Args:
            query: Search query
            k: Number of results
            field_filter: Optional field to filter by (e.g., 'Biology', 'AI')
        """
        # Encode query
        query_embedding = self.model.encode([query])[0].astype('float32')
        query_embedding = query_embedding / np.linalg.norm(query_embedding)
        
        # Search
        similarities, indices = self.index.search(
            query_embedding.reshape(1, -1), 
            len(self.papers) if field_filter else k
        )
        
        # Filter by field if requested
        results = []
        for idx, sim in zip(indices[0], similarities[0]):
            paper = self.papers[idx]
            if field_filter is None or paper['field'] == field_filter:
                results.append((paper, sim))
                if len(results) == k:
                    break
        
        return results
    
    def display_results(self, query: str, k: int = 5, field_filter: str = None):
        """Display search results in a nice format."""
        results = self.search(query, k, field_filter)
        
        print(f"\n{'='*80}")
        print(f"🔍 Query: '{query}'")
        if field_filter:
            print(f"   Filter: {field_filter} papers only")
        print(f"{'='*80}\n")
        
        for rank, (paper, similarity) in enumerate(results, 1):
            print(f"{rank}. 📄 {paper['title']}")
            print(f"   🏷️  {paper['field']} | 📊 Similarity: {similarity:.3f}")
            print(f"   📝 {paper['abstract'][:200]}...")
            print()

# Create search engine
search_engine = PaperSearchEngine(papers, embeddings, model)

print("✅ Search engine ready!")

## Step 6: Build a Simple Search Interface

In [ ]:
def compare_similarity_metrics(query: str, k: int = 5):
    """Compare different similarity metrics for the same query."""
    
    query_embedding = model.encode([query])[0].astype('float32')
    
    # L2 (Euclidean) distance - FAISS default
    index_l2 = faiss.IndexFlatL2(dimension)
    index_l2.add(embeddings)
    distances_l2, indices_l2 = index_l2.search(query_embedding.reshape(1, -1), k)
    
    # Inner product (cosine similarity when normalized)
    index_ip = faiss.IndexFlatIP(dimension)
    # Normalize embeddings
    embeddings_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    index_ip.add(embeddings_norm)
    query_norm = query_embedding / np.linalg.norm(query_embedding)
    similarities, indices_ip = index_ip.search(query_norm.reshape(1, -1), k)
    
    print(f"Query: '{query}'\n")
    
    # Create comparison dataframe
    comparison = []
    for rank in range(k):
        comparison.append({
            'Rank': rank + 1,
            'L2 Distance': titles[indices_l2[0][rank]],
            'L2 Score': f"{distances_l2[0][rank]:.3f}",
            'Cosine Similarity': titles[indices_ip[0][rank]],
            'Cos Score': f"{similarities[0][rank]:.3f}"
        })
    
    df = pd.DataFrame(comparison)
    print(df.to_string(index=False))
    
compare_similarity_metrics("machine learning for biology", k=5)

## Step 5: Compare Similarity Metrics

Vector databases use different distance metrics. Let's compare them!

In [ ]:
search_papers("gene editing and cancer treatment", k=3, use_db="faiss")

### Query 3: Gene Therapy Research

In [ ]:
search_papers("reducing carbon emissions and global warming", k=3, use_db="numpy")

### Query 2: Climate Change Solutions

In [ ]:
search_papers("artificial intelligence for medical diagnosis", k=3, use_db="faiss")

### Query 1: AI for Healthcare

In [ ]:
def search_papers(query: str, k: int = 5, use_db: str = "faiss"):
    """
    Search for relevant papers using semantic similarity.
    
    Args:
        query: Natural language search query
        k: Number of results to return
        use_db: Which database to use ('faiss', 'chroma', or 'numpy')
    """
    # Encode query
    query_embedding = model.encode([query])[0].astype('float32')
    
    print(f"🔍 Query: '{query}'\n")
    print("=" * 80)
    
    if use_db == "faiss":
        # FAISS search
        distances, indices = faiss_index.search(
            query_embedding.reshape(1, -1), k
        )
        
        for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), 1):
            print(f"\n{rank}. {titles[idx]}")
            print(f"   Field: {fields[idx]} | Distance: {dist:.3f}")
            print(f"   {abstracts[idx][:150]}...")
    
    elif use_db == "chroma" and chroma_available:
        # ChromaDB search
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=k
        )
        
        for rank, (doc, meta, dist) in enumerate(
            zip(results['documents'][0], 
                results['metadatas'][0],
                results['distances'][0]), 1
        ):
            print(f"\n{rank}. {meta['title']}")
            print(f"   Field: {meta['field']} | Distance: {dist:.3f}")
            print(f"   {doc[:150]}...")
    
    elif use_db == "numpy":
        # NumPy search
        results = simple_db.search(query_embedding, k=k)
        
        for rank, (idx, similarity) in enumerate(results, 1):
            print(f"\n{rank}. {titles[idx]}")
            print(f"   Field: {fields[idx]} | Similarity: {similarity:.3f}")
            print(f"   {abstracts[idx][:150]}...")
    
    print("\n" + "=" * 80)

## Step 4: Search with Natural Language Queries

Now the fun part! Let's search for papers using natural language queries.

In [ ]:
class SimpleVectorDB:
    """A minimal vector database implementation using NumPy."""
    
    def __init__(self, embeddings: np.ndarray, metadata: List[dict]):
        self.embeddings = embeddings
        self.metadata = metadata
        
    def search(self, query_embedding: np.ndarray, k: int = 5) -> List[Tuple[int, float]]:
        """
        Find k nearest neighbors using cosine similarity.
        
        Args:
            query_embedding: Query vector
            k: Number of results to return
            
        Returns:
            List of (index, similarity_score) tuples
        """
        # Normalize embeddings for cosine similarity
        query_norm = query_embedding / np.linalg.norm(query_embedding)
        db_norm = self.embeddings / np.linalg.norm(self.embeddings, axis=1, keepdims=True)
        
        # Compute cosine similarity
        similarities = db_norm @ query_norm
        
        # Get top k indices
        top_k_indices = np.argsort(similarities)[::-1][:k]
        
        return [(idx, similarities[idx]) for idx in top_k_indices]

# Create our custom database
simple_db = SimpleVectorDB(embeddings, papers)

print("✅ Simple NumPy vector DB created!")
print("   This is how vector databases work under the hood!")

### Option 3: Pure NumPy (Educational)

In [ ]:
# Install chromadb if needed: pip install chromadb
try:
    import chromadb
    from chromadb.utils import embedding_functions
    
    # Create in-memory ChromaDB collection
    chroma_client = chromadb.Client()
    collection = chroma_client.create_collection(
        name="research_papers",
        metadata={"description": "Academic paper abstracts"}
    )
    
    # Add documents
    collection.add(
        embeddings=embeddings.tolist(),
        documents=abstracts,
        metadatas=[{"title": t, "field": f} for t, f in zip(titles, fields)],
        ids=[f"paper_{i}" for i in range(len(papers))]
    )
    
    print(f"✅ ChromaDB collection created!")
    print(f"   Total documents: {collection.count()}")
    chroma_available = True
except ImportError:
    print("⚠️  ChromaDB not installed. Install with: pip install chromadb")
    chroma_available = False

### Option 2: ChromaDB (Easy & Lightweight)

In [ ]:
# Create FAISS index
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)  # L2 (Euclidean) distance
faiss_index.add(embeddings)

print(f"✅ FAISS index created!")
print(f"   Total vectors: {faiss_index.ntotal}")
print(f"   Dimension: {dimension}")

## Step 3: Build Vector Databases

We'll implement **three different approaches** to show you options:

1. **FAISS** - Meta's ultra-fast similarity search (best for production)
2. **ChromaDB** - Simple, lightweight, and easy to use
3. **NumPy** - DIY approach to understand the fundamentals

### Option 1: FAISS (Production-Ready)

In [ ]:
# Reduce dimensions to 2D for visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
embeddings_2d = tsne.fit_transform(embeddings)

# Create a color map for fields
field_colors = {
    'AI': '#FF6B6B',
    'AI-Biology': '#4ECDC4',
    'AI-Climate': '#95E1D3',
    'AI-Robotics': '#F38181',
    'Biology': '#AA96DA',
    'Climate': '#FCBAD3',
    'Economics': '#FFD93D'
}
colors = [field_colors[f] for f in fields]

# Plot
plt.figure(figsize=(14, 8))
for field in set(fields):
    mask = [f == field for f in fields]
    plt.scatter(
        embeddings_2d[mask, 0], 
        embeddings_2d[mask, 1],
        c=[field_colors[field]], 
        label=field, 
        s=200, 
        alpha=0.7,
        edgecolors='black',
        linewidth=1.5
    )

plt.title('Research Paper Embeddings (t-SNE Visualization)', fontsize=16, fontweight='bold')
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📊 Similar papers cluster together in embedding space!")

### Visualize the Embeddings

Let's use t-SNE to visualize these high-dimensional embeddings in 2D and see if similar papers cluster together!